# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using the `mlcroissant` library.

### Dataset Source
The dataset's Croissant schema is publicly accessible at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** The Croissant schema defines one or more record sets. All references use their full `@id`.


In [ ]:
# Find all record sets and display their fields/columns by @id
print("All RecordSets in dataset:")
record_sets = []
for rs in metadata.record_sets:
    record_sets.append(rs['@id'])
    print(f"- RecordSet @id: {rs['@id']}")
    if 'fields' in rs:
        print("  Fields (by @id):")
        for field in rs['fields']:
            fid = field['@id']
            print(f"    - {fid}")
            if 'column' in field:
                col_id = field['column']['@id'] if isinstance(field['column'], dict) else field['column']
                print(f"      (column: {col_id})")
    if 'columns' in rs and not rs.get('fields'):
        print("  Columns (by @id):")
        for col in rs['columns']:
            cid = col['@id']
            print(f"    - {cid}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id` values from the overview above.

In [ ]:
# If you have multiple record sets, list their @id's here
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

# Load each record set into a DataFrame
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set: {rs_id}")
    else:
        print(f"No records loaded for record set: {rs_id}")

# Show columns for the main record set (use the first by default)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Columns for record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic EDA steps:
- Filter records based on a numeric field (by `@id`).
- Normalize the values.
- Group or aggregate data.

**Note:** Refer to columns by their `@id` where possible.

In [ ]:
# Get the main record set's DataFrame
df = dataframes[main_rs_id]

# Show all field IDs for selection
print(f"Columns in use (should be @id):\n{df.columns.tolist()}")

# Example: pick first numeric field by checking dtype
numeric_candidates = df.select_dtypes(include='number').columns
if len(numeric_candidates) == 0:
    print("No numeric fields found for filtering.")
else:
    numeric_field = numeric_candidates[0]  # Choose the first numeric field
    print(f"Using numeric field '@id': {numeric_field}")

    threshold = df[numeric_field].mean()  # Use mean as threshold example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by the first non-numeric field
    non_numeric_candidates = df.select_dtypes(exclude='number').columns
    if len(non_numeric_candidates) > 0:
        group_field = non_numeric_candidates[0]
        print(f"Grouping by categorical '@id' field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No suitable non-numeric fields for grouping found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example using Matplotlib to plot a histogram and, if available, a boxplot by a group field.

In [ ]:
import matplotlib.pyplot as plt

if len(df.select_dtypes(include='number').columns) > 0:
    num_field = df.select_dtypes(include='number').columns[0]
    plt.figure(figsize=(7, 4))
    plt.hist(df[num_field].dropna(), bins=30, color='skyblue', edgecolor='black')
    plt.title(f'Histogram of {num_field}')
    plt.xlabel(num_field)
    plt.ylabel('Count')
    plt.show()

    # If grouping available, plot boxplot by group field
    nonnum_cols = df.select_dtypes(exclude='number').columns
    if len(nonnum_cols) > 0:
        grp_field = nonnum_cols[0]
        plt.figure(figsize=(10, 5))
        df.boxplot(column=num_field, by=grp_field)
        plt.title(f'Boxplot of {num_field} grouped by {grp_field}')
        plt.suptitle("")
        plt.xlabel(grp_field)
        plt.ylabel(num_field)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² mass spectrometry dataset, examined its structure, and conducted some exploratory analysis using `mlcroissant`. We used strict `@id` referencing for all entities, making this notebook robust and evolvable. For advanced analysis or machine learning, you can further process the DataFrames created here or integrate with other data science tools.